# Promoter Behavioral Localization in Baseline DNABERT-2

## Objective

Determine which sequence properties drive promoter-associated representations in baseline, non-fine-tuned DNABERT-2.

Using the Layer-5 linear probe established in Notebook 1, this study compares intact sequences with controlled corruptions designed to separate:

1. global nucleotide composition,
2. short-range sequence order,
3. regional position,
4. motif identity,
5. broader promoter context.

This is a behavioral-localization study. It identifies a stable clean-versus-corrupt effect before any head-, MLP-, or subspace-level intervention.

Cell 1 — Install and import dependencies

In [ ]:
!pip install -q datasets joblib

from pathlib import Path
from collections import Counter
import random
import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Torch version:", torch.__version__)

Device: cuda
Torch version: 2.11.0+cu128


2 — load GUE and check Notebook 1 artifacts

In [ ]:
from datasets import load_dataset

DATASET_NAME = "leannmlindsey/GUE"
DATASET_CONFIG = "prom_core_all"

ARTIFACT_DIR = Path("outputs/promoter_baseline_pilot")
PROBE_PATH = ARTIFACT_DIR / "dnabert_layer5_probe.joblib"

print("Loading GUE promoter dataset...")

promoter_dataset = load_dataset(
    DATASET_NAME,
    name=DATASET_CONFIG,
)

print(promoter_dataset)

print("\nChecking Notebook 1 artifacts...")
print("Artifact directory exists:", ARTIFACT_DIR.exists())
print("Layer-5 probe exists:", PROBE_PATH.exists())

if PROBE_PATH.exists():
    layer5_probe = joblib.load(PROBE_PATH)
    print("Layer-5 probe loaded successfully.")
else:
    print(
        "\nThe saved probe is not available in this runtime. "
        "We will need to upload it, mount Google Drive, "
        "or recreate it from Notebook 1."
    )

Loading GUE promoter dataset...


README.md:   0%|          | 0.00/8.47k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/438k [00:00<?, ?B/s]

dev.csv:   0%|          | 0.00/438k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47356 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5920 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5920 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'label'],
        num_rows: 47356
    })
    test: Dataset({
        features: ['sequence', 'label'],
        num_rows: 5920
    })
    dev: Dataset({
        features: ['sequence', 'label'],
        num_rows: 5920
    })
})

Checking Notebook 1 artifacts...
Artifact directory exists: False
Layer-5 probe exists: False

The saved probe is not available in this runtime. We will need to upload it, mount Google Drive, or recreate it from Notebook 1.


3 — mount Drive and look for the saved artifacts

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert/"
    "outputs/promoter_baseline_pilot"
)

DRIVE_PROBE_PATH = (
    DRIVE_ARTIFACT_DIR / "dnabert_layer5_probe.joblib"
)

print("Drive artifact directory:", DRIVE_ARTIFACT_DIR)
print("Directory exists:", DRIVE_ARTIFACT_DIR.exists())
print("Probe exists:", DRIVE_PROBE_PATH.exists())

if DRIVE_PROBE_PATH.exists():
    layer5_probe = joblib.load(DRIVE_PROBE_PATH)
    print("Layer-5 probe loaded successfully from Google Drive.")
else:
    print(
        "\nThe probe has not yet been copied to this Drive folder."
    )

Mounted at /content/drive
Drive artifact directory: /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_baseline_pilot
Directory exists: False
Probe exists: False

The probe has not yet been copied to this Drive folder.


4 — create the persistent folder and reproduce the same pilot subsets

In [ ]:
from datasets import Dataset

DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_PER_CLASS = 1000
DEV_PER_CLASS = 400
TEST_PER_CLASS = 400


def balanced_subset(dataset_split, per_class, seed):
    df = dataset_split.to_pandas()

    sampled_parts = []

    for label in sorted(df["label"].unique()):
        sampled = (
            df[df["label"] == label]
            .sample(
                n=per_class,
                random_state=seed,
                replace=False,
            )
        )

        sampled_parts.append(sampled)

    subset_df = (
        pd.concat(sampled_parts)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return Dataset.from_pandas(
        subset_df,
        preserve_index=False,
    )


pilot_train = balanced_subset(
    promoter_dataset["train"],
    TRAIN_PER_CLASS,
    SEED,
)

pilot_dev = balanced_subset(
    promoter_dataset["dev"],
    DEV_PER_CLASS,
    SEED,
)

pilot_test = balanced_subset(
    promoter_dataset["test"],
    TEST_PER_CLASS,
    SEED,
)

print("Drive folder created:", DRIVE_ARTIFACT_DIR.exists())
print("Pilot train:", len(pilot_train))
print("Pilot dev:", len(pilot_dev))
print("Pilot test:", len(pilot_test))
print("Train labels:", Counter(pilot_train["label"]))

Drive folder created: True
Pilot train: 2000
Pilot dev: 800
Pilot test: 800
Train labels: Counter({1: 1000, 0: 1000})


# Cell 5A — Pin DNABERT-2-compatible Hugging Face versions

In [ ]:
!pip uninstall -y transformers tokenizers huggingface_hub accelerate

!pip install -q \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    huggingface-hub==0.24.7\
    accelerate==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 31.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.24.7 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.24.7 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.24.7 which is incompatible.


5 — load DNABERT-2 and capture only Layer 5

In [ ]:
import gc
import torch

from transformers import AutoTokenizer, AutoModel
from transformers.models.bert.configuration_bert import BertConfig

MODEL_NAME = "quietflamingo/dnabert2-no-flashattention"
LAYER_INDEX = 4  # Zero-based index: transformer Layer 5

# Clear any partial model objects.
for variable_name in ["backbone", "config"]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()

# Construct the custom model safely on CPU first.
torch.set_default_device("cpu")

print("CUDA available:", torch.cuda.is_available())
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Loading configuration...")

config = BertConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = tokenizer.pad_token_id
config.output_hidden_states = False
config.return_dict = False

print("Constructing model on CPU...")

backbone = AutoModel.from_pretrained(
    MODEL_NAME,
    config=config,
    trust_remote_code=True,
    low_cpu_mem_usage=False,
    device_map=None,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Moving model to:", device)

backbone = backbone.to(device)
backbone.eval()

# Capture only Layer 5.
captured_layer5 = {}

def layer5_hook(module, inputs, output):
    hidden = output[0] if isinstance(output, tuple) else output
    captured_layer5["hidden"] = hidden.detach()

layer5_hook_handle = (
    backbone.encoder.layer[LAYER_INDEX]
    .register_forward_hook(layer5_hook)
)

print("\nModel loaded successfully.")
print("Model class:", backbone.__class__.__name__)
print("Hidden size:", backbone.config.hidden_size)
print("Selected layer:", LAYER_INDEX + 1)
print("Model device:", next(backbone.parameters()).device)

CUDA available: True
Loading tokenizer...


config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-no-flashattention:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

...

[transformers] A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-no-flashattention:
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-no-flashattention:
- bert_layers.py
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  468MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

/root/.cache/huggingface/modules/transformers_modules/quietflamingo/dnabert2_hyphen_no_hyphen_flashattention/813031b2bf86d9e960a027e2734c908009f31601/bert_layers.py:123: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(


RuntimeError: Tensor on device meta is not on the expected device cpu!